# <center>  House Price Prediction 
<img src="/assets/house.jpg" width="500">

##  <center> <span style='background:orange'>  Feature Selection  </span>

 <p> In ou dataset we have more tha 70 features/columns, so its a big number and its diffuclt
to analyse all of those features, also, among theme there are some non informative 
or irrelevant features, so is more practicall to remove theme to reduce the shape of our dataset. </p>

 In the this part of our project, we will select the most relevant features. for this purpose, 
 we will use the *filter_based* mothod.
* to select **the numerical variables**, we choose to apply:
   * the **Pearson's Correlation Coefficient** method 
    via the  <span style='background:orange'> </span>
    `f_regression()` function.
   * the **Spearman Rank Coefficient** method via the  <span style='background:orange'> </span>
    `corrwith()` Pandas function.
   * the **Mutual Gain Information** method via the  <span style='background:orange'> </span>
    `mutual_info_regression()` function.
 * to select the **categorical variables**, we choose to apply:
    * the **ANOVA Correlation Coefficient** method.
    * **Information Gain** method. 
We will compare the results and finaly choose the best features.
** NB : The selected features still provisinate until the are test 

In [3]:
### load libraries
import pandas as pd
import numpy as np
from pathlib import Path

#data viz tools
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

#feature selection tools
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import chi2

# Encoding Tools
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
# skip warnings
import warnings
warnings.filterwarnings("ignore")

In [8]:
# Load the dataset
df = pd.read_csv('../datasets/house_price_cleaned.csv')
df.head(10)

,Order,PID,MS SubClass,MS Zoning,Lot Area,Street,Lot Shape,Land Contour,Utilities,Lot Config,...,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,31770.0,Pave,IR1,Lvl,AllPub,Corner,...,0,0,0.0,0,0,5,2010,WD,Normal,215000.0
1,2,526350040,20,RH,11622.0,Pave,Reg,Lvl,AllPub,Inside,...,0,0,120.0,0,0,6,2010,WD,Normal,105000.0
2,3,526351010,20,RL,14267.0,Pave,IR1,Lvl,AllPub,Corner,...,0,0,0.0,0,971,6,2010,WD,Normal,172000.0
3,4,526353030,20,RL,11160.0,Pave,Reg,Lvl,AllPub,Corner,...,0,0,0.0,0,0,4,2010,WD,Normal,244000.0
4,5,527105010,60,RL,13830.0,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0.0,0,0,3,2010,WD,Normal,189900.0
5,6,527105030,60,RL,9978.0,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0.0,0,0,6,2010,WD,Normal,195500.0
6,7,527127150,120,RL,4920.0,Pave,Reg,Lvl,AllPub,Inside,...,170,0,0.0,0,0,4,2010,WD,Normal,213500.0
7,8,527145080,120,RL,5005.0,Pave,IR1,HLS,AllPub,Inside,...,0,0,144.0,0,0,1,2010,WD,Normal,191500.0
8,9,527146030,120,RL,5389.0,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0.0,0,0,3,2010,WD,Normal,236500.0
9,10,527162130,60,RL,7500.0,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0.0,0,0,6,2010,WD,Normal,189000.0


In [9]:
 # Droping the Order and PID columns as they are not useful for prediction
df.drop(columns=["Order", "PID"], inplace=True)
numerical_features= df.select_dtypes(include=["int64", "float64"])
numerical_features

,MS SubClass,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,...,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,SalePrice
0,20,31770.0,6,5,1960,1960,112.0,639.0,0.0,441.0,...,210.0,62.0,0,0,0.0,0,0,5,2010,215000.0
1,20,11622.0,5,6,1961,1961,0.0,468.0,144.0,270.0,...,140.0,0.0,0,0,120.0,0,0,6,2010,105000.0
2,20,14267.0,6,6,1958,1958,108.0,923.0,0.0,406.0,...,393.0,36.0,0,0,0.0,0,971,6,2010,172000.0
3,20,11160.0,7,5,1968,1968,0.0,1065.0,0.0,1045.0,...,0.0,0.0,0,0,0.0,0,0,4,2010,244000.0
4,60,13830.0,5,5,1997,1998,0.0,791.0,0.0,137.0,...,212.0,34.0,0,0,0.0,0,0,3,2010,189900.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,80,7937.0,6,6,1984,1984,0.0,819.0,0.0,184.0,...,120.0,0.0,0,0,0.0,0,0,3,2006,142500.0
2926,20,8885.0,5,5,1983,1983,0.0,301.0,324.0,239.0,...,164.0,0.0,0,0,0.0,0,0,6,2006,131000.0
2927,85,10441.0,5,5,1992,1992,0.0,337.0,0.0,575.0,...,80.0,32.0,0,0,0.0,0,700,7,2006,132000.0
2928,20,10010.0,5,5,1974,1975,0.0,1071.0,123.0,195.0,...,240.0,38.0,0,0,0.0,0,0,4,2006,170000.0


In [10]:
numerical_features.isnull().sum()

MS SubClass        0
Lot Area           0
Overall Qual       0
Overall Cond       0
Year Built         0
Year Remod/Add     0
Mas Vnr Area       0
BsmtFin SF 1       0
BsmtFin SF 2       0
Bsmt Unf SF        0
Total Bsmt SF      0
1st Flr SF         0
2nd Flr SF         0
Low Qual Fin SF    0
Gr Liv Area        0
Bsmt Full Bath     0
Bsmt Half Bath     0
Full Bath          0
Half Bath          0
Bedroom AbvGr      0
Kitchen AbvGr      0
TotRms AbvGrd      0
Fireplaces         0
Garage Yr Blt      0
Garage Cars        0
Garage Area        0
Wood Deck SF       0
Open Porch SF      0
Enclosed Porch     0
3Ssn Porch         0
Screen Porch       0
Pool Area          0
Misc Val           0
Mo Sold            0
Yr Sold            0
SalePrice          0
dtype: int64